# Leak minority turn: does composition predict the handedness lag?

## Question and why it matters
On a compact ant, a partial leak of random turns on fresh ground pushes growth from compact towards porous. The leak-handedness
cohort (`experiments/active/2026-11-leak-handedness-cohort/`, 77 rules) found that whether the forced fresh-ground turn is R or L
changes how much leak a rule tolerates, strongly and rule by rule, in both directions. After the fact, the direction seemed to track
composition: the forced turn did better when it was the rule's *minority* letter (Spearman +0.40). This notebook scores that pattern
as a registered prediction on 81 new rules. Registration, gates and predictions: `README.md`; lab record: `NOTEBOOK.md`.

## Assumptions and definitions
* Leak EF: on a never-departed cell the forced base turn is replaced by R or L with probability 1/2 each (20-slot `fs:` set).
  Arm R = base R (the ant's colour-0 turn), arm L = base L.
* Early a: slope of log area on log t over k = 12..22 (probe, board 16384).
* Rise fraction F(EF) = (a(EF) - a(0)) / (a(1) - a(0)) per arm; lag Lambda = mean over EF 0.3, 0.5 of F_R - F_L.
  Positive: base L tolerates the leak better.
* R fraction fr = share of R letters in the rule string.

## Provenance
Git commit, date and SHA-256 of every input. Seeds 2026111301..04 (held-out), 2026111201..04 (anchor).

In [1]:
import hashlib, subprocess, datetime
from pathlib import Path
import numpy as np
import polars as pl

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = ROOT / "experiments/active/2026-11-leak-minority-turn"
INPUTS = [EXP / f"results/{n}" for n in ("rules.txt", "specs.txt", "probe.csv", "runs.csv", "rules.csv",
                                         "analysis.txt", "calibration.txt", "posthoc.txt", "provenance.txt")]
sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("git", sha, "| run", datetime.date.today().isoformat())
for p in INPUTS:
    print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))

git 359fc84 | run 2026-11-13
cd5e2136b5d526d7 experiments/active/2026-11-leak-minority-turn/results/rules.txt
ccd365fd86aa0077 experiments/active/2026-11-leak-minority-turn/results/specs.txt
7bb92b3c698e6579 experiments/active/2026-11-leak-minority-turn/results/probe.csv
3be157fefbf43d1c experiments/active/2026-11-leak-minority-turn/results/runs.csv
17b72abe6bfe0239 experiments/active/2026-11-leak-minority-turn/results/rules.csv
706e23e2dd64834a experiments/active/2026-11-leak-minority-turn/results/analysis.txt
e9b1059cee05b351 experiments/active/2026-11-leak-minority-turn/results/calibration.txt
f41ab6dd94bd5930 experiments/active/2026-11-leak-minority-turn/results/posthoc.txt
2db4c8f0794a2136 experiments/active/2026-11-leak-minority-turn/results/provenance.txt


## Calibration (synthetic, before the run)

In [2]:
print((EXP / "results/calibration.txt").read_text())

# dev cohort C (n 77): b_dev +1.224 per unit R fraction, residual sd 0.350; dev Spearman +0.404
# held-out R fractions from 110 drawn rules; 1000 draws, 2000 bootstrap resamples, seed 20261113
n 50 slope b_dev: P1 replicates 0.812, unresolved 0.188 | P2 holds 0.686, unresolved 0.314
n 50 slope b_dev/2: P1 fails 0.012, replicates 0.302, unresolved 0.686 | P2 fails 0.015, holds 0.235, unresolved 0.750
n 50 slope 0: P1 fails 0.286, replicates 0.028, unresolved 0.686 | P2 fails 0.238, holds 0.029, unresolved 0.733
n 70 slope b_dev: P1 replicates 0.935, unresolved 0.065 | P2 holds 0.828, unresolved 0.172
n 70 slope b_dev/2: P1 fails 0.017, replicates 0.431, unresolved 0.552 | P2 fails 0.020, holds 0.327, unresolved 0.653
n 70 slope 0: P1 fails 0.386, replicates 0.020, unresolved 0.594 | P2 fails 0.321, holds 0.017, unresolved 0.662
n 90 slope b_dev: P1 replicates 0.969, unresolved 0.031 | P2 holds 0.900, unresolved 0.100
n 90 slope b_dev/2: P1 fails 0.014, replicates 0.528, unresolved 0.458

If the development slope were real, the registered rule would essentially never say "fails"; under no slope it says
"replicates" about 2-3% of the time.

## Gates and registered scores
The frozen scorer's output, unedited.

In [3]:
print((EXP / "results/analysis.txt").read_text())

# leak minority-turn scoring; 3016 specs
G1 replication: 156/156 anchor rows identical to leak-handedness -> PASS
cohort H: 110 rules, eligible 84
  ineligible: RLLLLRRLL (R intermediate, L compact), RLLLLRRLRLLR (R compact, L escape), RLLLLRRRLR (R compact, L escape), RLLLRLLLRRRL (R compact, L filler), RLLRLLLRRLLR (R compact, L escape), RLLRLLLRRLR (R compact, L escape), RLLRLRLLLLRR (R compact, L escape), RLRLLLLLRR (R compact, L filler), RLRLLLLRLLL (R compact, L filler), RLRLLLLRLRLL (R compact, L filler), RLRLLLRRRR (R compact, L intermediate), RLRLLRRRRLR (R compact, L escape), RLRRLRLRRL (R compact, L escape), RLRRLRRRRR (R compact, L intermediate), RLRRRLLLLRRL (R compact, L escape), RLRRRLRLLLLR (R compact, L escape), RRLLLLLRLR (R compact, L filler), RRLRLRLRRR (R compact, L intermediate), RRLRLRRL (R compact, L escape), RRLRLRRLRLLR (R compact, L escape), RRLRRRLLRLLL (R compact, L escape), RRLRRRLRRL (R compact, L escape), RRRLLLRLR (R compact, L intermediate), RRRLLRLLLL

Both gates pass. **P1, P2 and P3 all fail.** The held-out correlation is slightly negative and its interval excludes the
development value.

## Composition against lag, held-out rules

In [4]:
r = pl.read_csv(EXP / "results/rules.csv").filter(pl.col("scoreable")).sort("fr")
for lo, hi in [(0, .4), (.4, .45), (.45, .55), (.55, .6), (.6, .7), (.7, 1.01)]:
    m = r.filter((pl.col("fr") >= lo) & (pl.col("fr") < hi))
    if m.height:
        lam = m["lam"].to_numpy()
        print(f"fr [{lo:.2f}, {min(hi, 1):.2f}): n {m.height:2d}  median Lambda {np.median(lam):+.3f}  " + "".join("+" if v > 0 else "-" for v in lam))

fr [0.00, 0.40): n  6  median Lambda -0.171  ++----
fr [0.40, 0.45): n 13  median Lambda -0.142  --+++--+-++--
fr [0.45, 0.55): n 21  median Lambda -0.061  --+--+--+-++++--+-+-+
fr [0.55, 0.60): n 18  median Lambda +0.090  --++---+++--++++++
fr [0.60, 0.70): n  9  median Lambda -0.042  +----+++-
fr [0.70, 1.00): n 14  median Lambda -0.240  ---+-+---++---


## Post hoc (not registered): the spread itself replicates

In [5]:
print((EXP / "results/posthoc.txt").read_text())

# POST HOC, not registered. Cohort H, 81 scoreable rules.
Lambda: mean -0.079, sd 0.381, quartiles -0.360 / +0.225; |Lambda| > 0.4 in 27 rules
median within-rule per-seed sd of early a at EF 0.3/0.5: 0.0209; median Delta_R 0.182; implied seed noise on one rule's Lambda ~ 0.057



The lag is as large and as rule-specific as in the development cohort (sd 0.38 against ~0.06 seed noise), so handedness
matters; what failed is the explanation of its sign by letter counts.

## Caveats
* Finite window (k = 12..22), board 16384, 4 seeds per cell, two leak levels.
* Core class only; 26 of 110 drawn rules were ineligible (a forced-L base already escapes, fills or is intermediate).
* "Fails" covers R fraction and the two prefix groups as registered, not every feature of the rule table.

## Next steps
Look for a predictor of the sign in the dynamics rather than the letters (registered on the 77 + 81 rules, scored on a fresh draw),
or permute letters within one rule's table to hold composition fixed.